# Jalon B1 - Interventions Demo

Ce notebook montre l'usage des outils B1: optimisation d'un delta latent, application en hook, protocole mono-cible et sweep de couches.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from boardGPT.nn import GPT, GPTConfig
from boardGPT.interpretability import (
    LinearBoardProbe,
    InterventionTarget,
    InterventionConfig,
    optimize_residual_delta,
    run_probe_intervention,
    ProtocolConfig,
    run_single_target_protocol,
    run_layer_sweep_protocol,
    extract_residuals,
)

In [ ]:
# Tiny model pour demo locale
cfg = GPTConfig(block_size=8, vocab_size=25, n_layer=2, n_head=1, n_embd=16, dropout=0.0, bias=True)
model = GPT(cfg, use_flash=False)

# Probe cible artificiellement controllable (cell=5, class=2)
probe = LinearBoardProbe(input_dim=16)
with torch.no_grad():
    probe.proj.weight.zero_()
    probe.proj.bias.zero_()
    base = 5 * 3
    probe.proj.weight[base + 0, 0] = -2.0
    probe.proj.weight[base + 1, 0] = -1.0
    probe.proj.weight[base + 2, 0] = 2.0

In [ ]:
tokens = torch.randint(0, cfg.vocab_size, (1, cfg.block_size))
target = InterventionTarget(cell_index=5, target_class=2, position='last')
intv_cfg = InterventionConfig(steps=40, lr=0.08, reg_strength=1e-4, device='cpu')

# Extraction d'une activation de base
base_res = extract_residuals(model, tokens, layer=0, device='cpu')
base_vec = base_res[0, -1, :]

opt = optimize_residual_delta(probe, base_vec, target, intv_cfg)
print('before target prob:', opt['before_target_prob'])
print('after target prob :', opt['after_target_prob'])
print('gain            :', opt['target_prob_gain'])

In [ ]:
# Run end-to-end sur une sequence
single = run_probe_intervention(
    model=model,
    probe=probe,
    token_batch=tokens,
    layer=0,
    target=target,
    config=intv_cfg,
    device='cpu',
)

print('before prob:', single['before_target_prob'])
print('after prob :', single['after_target_prob'])
print('gain      :', single['target_prob_gain'])
print('success   :', single['success'])

In [ ]:
# Protocole mono-cible sur plusieurs exemples
batch_tokens = torch.randint(0, cfg.vocab_size, (8, cfg.block_size))
loader = DataLoader(TensorDataset(batch_tokens), batch_size=2, shuffle=False)
pconf = ProtocolConfig(max_examples=6, intervention_steps=20, intervention_lr=0.08, intervention_reg_strength=1e-4, device='cpu')

report = run_single_target_protocol(
    model=model,
    probe=probe,
    batches=loader,
    layer=0,
    target_cell=5,
    target_class=2,
    position='last',
    config=pconf,
)
report['summary']

In [ ]:
# Sweep de couches
sweep = run_layer_sweep_protocol(
    model=model,
    probe=probe,
    batches=loader,
    layers=[0, 1],
    target_cell=5,
    target_class=2,
    position='last',
    config=pconf,
)
for layer, data in sweep['layer_results'].items():
    print('layer', layer, '->', data['summary'])

Tu peux ensuite remplacer le tiny model/probe par ton modèle BoardGPT + probe entraîné sur données réelles pour des expériences B1 complètes.